# 02 · Feature-relation audit and detectors (NF-UNSW-NB15-v3 and NF-ToN-IoT-v3)

Design decisions:
- Inputs: the notebook 01 parquet files (`XIDS_Certify_Research/data/nf3_v1/`). Only the detector_train and development roles are read here; calibration and locked_test stay untouched.
- Relation audit: every candidate relation between NetFlow fields (durations, packet counts, byte bounds, the packet-size histogram, throughput, per-second byte fields, TCP flags, TTL order, inter-arrival statistics, retransmissions) is checked on detector_train rows. The attacker operation model in notebook 03 may use a relation only if it holds on at least 99.9% of applicable rows in both datasets. Field names alone are not trusted.
- Detector: one XGBoost model per dataset (300 trees at most, depth 6, early stopping on development), trained on distinct records without weights. The saved model is trimmed to its best round, so the verified model and the deciding model are the same object.
- Operating threshold: the score giving a 1% false-alarm rate on development benign flows, fixed here and reused unchanged later.
- Metrics on development, reported per distinct record and weighted by multiplicity (how often each record occurred in the raw traffic).
- Shortcut check: how well each feature separates the classes on its own; a feature near 1.0 is reported as a shortcut risk.
- Models are saved outside the repository in `XIDS_Certify_Research/models/nf02/`; their SHA-256 and the thresholds are committed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, importlib, json, os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import sklearn
import xgboost
import src.nf3_data as nf
import src.relations as rel
import src.detector as det
for m in (nf, rel, det):
    importlib.reload(m)

TABLES = REPO / 'results' / 'tables'
MODELS = PROJECT / 'models' / 'nf02'
DATASETS = ['unsw', 'ton']
FEATS = nf.MODEL_FEATURES
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 70)
VERSIONS = {'pandas': pd.__version__, 'numpy': np.__version__, 'xgboost': xgboost.__version__, 'sklearn': sklearn.__version__}
print(VERSIONS)
summary01 = json.loads((TABLES / 'nf01_summary.json').read_text())
print('data split hashes:', summary01['split_hash'])

## Feature-relation audit (detector_train rows only)

In [ ]:
audits = []
for ds in DATASETS:
    df = nf.load_split(PROJECT, ds, roles=['detector_train'], columns=FEATS + ['flow_start_ms', 'flow_end_ms'])
    audits.append(rel.audit(df, ds))
    print(f'{ds}: {len(df):,} rows audited')
    del df
    gc.collect()
relations = pd.concat(audits, ignore_index=True)
verdict = rel.verdict(relations)
display(verdict.sort_values('usable_in_operation_model', ascending=False))

In [ ]:
# ratio relations: the quantiles show the true scale factor when the named formula is wrong
display(relations[relations['kind'] == 'ratio = 1'][['dataset', 'relation', 'rows_applicable', 'share_satisfied', 'q01', 'q50', 'q99']])

## Detectors: train on detector_train, early stopping and threshold on development

In [ ]:
def run_detector(ds):
    cols = FEATS + ['y', 'family', 'multiplicity', 'role']
    df = nf.load_split(PROJECT, ds, roles=['detector_train', 'development'], columns=cols)
    tr, dv = df[df['role'] == 'detector_train'], df[df['role'] == 'development']
    booster = det.fit(tr[FEATS], tr['y'].to_numpy(), dv[FEATS], dv['y'].to_numpy())
    s_dev = det.predict(booster, dv[FEATS])
    y_dev = dv['y'].to_numpy()
    thr = det.threshold_at_fpr(s_dev[y_dev == 0])
    m_rec, fam_rec = det.metrics(y_dev, s_dev, thr, family=dv['family'].to_numpy())
    m_w, fam_w = det.metrics(y_dev, s_dev, thr, weight=dv['multiplicity'].to_numpy(), family=dv['family'].to_numpy())
    shortcut = det.single_feature_auroc(tr[FEATS], tr['y'].to_numpy()).assign(dataset=ds)
    imp = det.importance(booster, FEATS).assign(dataset=ds)
    path = MODELS / f'{ds}_xgb.json'
    sha = det.save(booster, path)
    stats = det.tree_stats(booster)
    rec = {'dataset': ds, 'model_path': str(path.relative_to(PROJECT)), 'model_sha256': sha, 'threshold': thr,
           'train_rows': int(len(tr)), 'development_rows': int(len(dv)), **stats, 'versions': VERSIONS}
    metrics = pd.DataFrame([{'dataset': ds, 'weighting': 'distinct records', **m_rec},
                            {'dataset': ds, 'weighting': 'multiplicity', **m_w}])
    fams = pd.concat([fam_rec.assign(dataset=ds, weighting='distinct records'),
                      fam_w.assign(dataset=ds, weighting='multiplicity')], ignore_index=True)
    del df, tr, dv
    gc.collect()
    return rec, metrics, fams, shortcut, imp

results = {}

In [ ]:
results['unsw'] = run_detector('unsw')
rec, metrics, fams, shortcut, imp = results['unsw']
print(json.dumps({k: v for k, v in rec.items() if k != 'versions'}, indent=2))
display(metrics)
display(fams)

In [ ]:
results['ton'] = run_detector('ton')
rec, metrics, fams, shortcut, imp = results['ton']
print(json.dumps({k: v for k, v in rec.items() if k != 'versions'}, indent=2))
display(metrics)
display(fams)

## Shortcut check: features that separate the classes on their own

In [ ]:
shortcuts = pd.concat([results[d][3] for d in DATASETS], ignore_index=True)
importances = pd.concat([results[d][4] for d in DATASETS], ignore_index=True)
for ds in DATASETS:
    s = shortcuts[shortcuts['dataset'] == ds]
    print(f"{ds}: features with AUROC alone >= 0.99: {s.loc[s['auroc_alone'] >= 0.99, 'feature'].tolist()}")
    display(s.head(10))
    display(importances[importances['dataset'] == ds].head(10))

## Tables, model hashes and configuration

In [ ]:
relations.to_csv(TABLES / 'nf02_relations.csv', index=False)
verdict.to_csv(TABLES / 'nf02_relations_verdict.csv', index=False)
pd.concat([results[d][1] for d in DATASETS], ignore_index=True).to_csv(TABLES / 'nf02_detector_metrics.csv', index=False)
pd.concat([results[d][2] for d in DATASETS], ignore_index=True).to_csv(TABLES / 'nf02_family_flag_rates.csv', index=False)
shortcuts.to_csv(TABLES / 'nf02_single_feature_auroc.csv', index=False)
importances.to_csv(TABLES / 'nf02_importance.csv', index=False)
models = {d: results[d][0] for d in DATASETS}
models['detector_config_sha256'] = det.write_config(REPO)
models['data_split_hash'] = summary01['split_hash']
models['usable_relations'] = sorted(verdict.loc[verdict['usable_in_operation_model'], 'relation'].tolist())
(TABLES / 'nf02_models.json').write_text(json.dumps(models, indent=2) + '\n')
display(pd.read_csv(TABLES / 'nf02_detector_metrics.csv'))
print('usable relations:', models['usable_relations'])

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 02: feature-relation audit on detector_train rows with a usability verdict for the attacker model, XGBoost detectors per dataset trimmed to the best round, development threshold at 1% false-alarm rate, metrics per distinct record and weighted by multiplicity, per-family flag rates, single-feature shortcut check, model hashes"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)